In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data.csv', sep=';')

#Shape
df.info()
df.isnull().sum().sort_values(ascending=False)

#Target column distribution
target_cols = [c for c in df.columns if 'Maladies diagnostiquées' in c]
print(target_cols)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Columns: 109 entries, Centre de santé to _uuid
dtypes: float64(12), object(97)
memory usage: 255.6+ KB
['Maladies diagnostiquées', 'Maladies diagnostiquées/Paludisme (Malaria)', 'Maladies diagnostiquées/Dengue', 'Maladies diagnostiquées/Chikunguya', 'Maladies diagnostiquées/Fièvre jaune (yellow fever)', 'Maladies diagnostiquées/Fièvre Typhoïde (Thyphoid fever)', 'Maladies diagnostiquées/Zika', 'Maladies diagnostiquées/Autres maladies diagnostiqué (Others diseases)', 'Maladies diagnostiquées/Option 8']


In [24]:
for col in diag_cols:
    print(col, "->", df[col].unique())

Maladies diagnostiquées -> ['Paludisme (Malaria) Dengue'
 'Paludisme (Malaria) Fièvre Typhoïde (Thyphoid fever)'
 'Paludisme (Malaria)'
 'Paludisme (Malaria) Autres maladies diagnostiqué (Others diseases)'
 'Paludisme (Malaria) Fièvre jaune (yellow fever)'
 'Dengue Fièvre jaune (yellow fever)' 'Dengue'
 'Dengue Autres maladies diagnostiqué (Others diseases)'
 'Paludisme (Malaria) Dengue Autres maladies diagnostiqué (Others diseases)'
 'Paludisme (Malaria) Fièvre jaune (yellow fever) Autres maladies diagnostiqué (Others diseases)'
 'Paludisme (Malaria) Fièvre Typhoïde (Thyphoid fever) Autres maladies diagnostiqué (Others diseases)'
 'Dengue Fièvre Typhoïde (Thyphoid fever)'
 'Autres maladies diagnostiqué (Others diseases)' nan
 'Paludisme (Malaria) Dengue Fièvre Typhoïde (Thyphoid fever)'
 'Fièvre jaune (yellow fever) Fièvre Typhoïde (Thyphoid fever)']
Maladies diagnostiquées/Paludisme (Malaria) -> [ 1.  0. nan]
Maladies diagnostiquées/Dengue -> [ 1.  0. nan]
Maladies diagnostiquées/Chi

In [25]:
# Sesuaikan mapping dengan value yang muncul
yes_values = ['Yes', 'Oui', '1', 1, True]

df[diag_cols] = df[diag_cols].apply(
    lambda col: col.map(lambda x: 1 if x in yes_values else (0 if pd.notna(x) else np.nan))
)

# Sekarang bisa di-sum
df['total_diag'] = df[diag_cols].sum(axis=1)
print(df['total_diag'].value_counts())

total_diag
2.0    149
1.0    141
3.0      9
0.0      1
Name: count, dtype: int64


In [26]:
# Drop kolom yang semua 0 / redundan
drop_target_cols = [
    'Maladies diagnostiquées',
    'Maladies diagnostiquées/Chikunguya',
    'Maladies diagnostiquées/Zika',
    'Maladies diagnostiquées/Option 8',
]
df.drop(columns=drop_target_cols, inplace=True)

# Target kolom yang valid
target_cols = [
    'Maladies diagnostiquées/Paludisme (Malaria)',
    'Maladies diagnostiquées/Dengue',
    'Maladies diagnostiquées/Fièvre jaune (yellow fever)',
    'Maladies diagnostiquées/Fièvre Typhoïde (Thyphoid fever)',
    'Maladies diagnostiquées/Autres maladies diagnostiqué (Others diseases)',
]

# Cek distribusi per label
print(df[target_cols].sum())

Maladies diagnostiquées/Paludisme (Malaria)                               270.0
Maladies diagnostiquées/Dengue                                             56.0
Maladies diagnostiquées/Fièvre jaune (yellow fever)                        12.0
Maladies diagnostiquées/Fièvre Typhoïde (Thyphoid fever)                   29.0
Maladies diagnostiquées/Autres maladies diagnostiqué (Others diseases)     99.0
dtype: float64


In [27]:
# Berapa baris yang semua target-nya nan?
print(df[target_cols].isna().all(axis=1).sum())

# Opsi: drop baris yang tidak ada label sama sekali
df = df[~df[target_cols].isna().all(axis=1)].copy()

# Isi nan yang tersisa dengan 0 (asumsi: tidak disebutkan = negatif)
df[target_cols] = df[target_cols].fillna(0).astype(int)

1
